# Automatic differentiation

In [1]:
using SciBmad
using DifferentiationInterface
import DifferentiationInterface as DI
# Different differentiation packages:
using ForwardDiff, GTPSA, ReverseDiff, FiniteDiff, FiniteDifferences

In [2]:
K1 = 0.36
K2 = 0.1

qf = Quadrupole(L=0.5, Kn1=K1)
sf = Sextupole(L=0.2, Kn2=K2)
d = Drift(L=0.6)
b = SBend(L=6.0, angle=π/132)
qd = Quadrupole(L=0.5, Kn1=-K1)
sd = Sextupole(L=0.2, Kn2=-K2)
fodo = Beamline([qf, d, sf, b, sd, d, qd], species_ref=Species("electron"), E_ref=18e9);

In [3]:
# Returns exit coordinates for 1 particle
function track_a_particle(v0, beamline)
    res = track(beamline; v0=reshape(v0, 1, 6))
    return res.v[:, :, end]
end
x0 = [1e-2, 0.0, 1e-2, 0.0, 0.0, 0.0];

In [4]:
# Forward AD using ForwardDiff.jl:
prep_FD = DI.prepare_jacobian(track_a_particle, AutoForwardDiff(), x0, Constant(fodo))
J_FD = DI.jacobian(track_a_particle, prep_FD, AutoForwardDiff(), x0, Constant(fodo))

6×6 Matrix{Float64}:
 -0.501416     8.58615      0.000478046  -0.0031526   0.0   0.122033
 -0.257407     2.41344     -0.000656686  -0.00360535  0.0   0.0439661
  0.00147842  -0.00193047   2.4167        8.59143     0.0  -0.0137771
  2.98994e-5  -0.00260515  -0.257077     -0.500127    0.0   0.00254321
 -0.00937121  -0.0829385   -0.00270552   -0.0152608   1.0  -0.000482959
  0.0          0.0          0.0           0.0         0.0   1.0

In [5]:
# Taylor AD using GTPSA.jl:
prep_GTPSA = DI.prepare_jacobian(track_a_particle, AutoGTPSA(), x0, Constant(fodo))
J_GTPSA = DI.jacobian(track_a_particle, prep_GTPSA, AutoGTPSA(), x0, Constant(fodo))

6×6 Matrix{Float64}:
 -0.501416     8.58615      0.000478046  -0.0031526   0.0   0.122033
 -0.257407     2.41344     -0.000656686  -0.00360535  0.0   0.0439661
  0.00147842  -0.00193047   2.4167        8.59143     0.0  -0.0137771
  2.98994e-5  -0.00260515  -0.257077     -0.500127    0.0   0.00254321
 -0.00937121  -0.0829385   -0.00270552   -0.0152608   1.0  -0.000482959
  0.0          0.0          0.0           0.0         0.0   1.0

In [6]:
# Reverse AD using ReverseDiff.jl:
prep_RD = DI.prepare_jacobian(track_a_particle, AutoReverseDiff(;compile=true), x0, Constant(fodo))
J_RD = DI.jacobian(track_a_particle, prep_RD, AutoReverseDiff(;compile=true), x0, Constant(fodo))

6×6 Matrix{Float64}:
 -0.501416     8.58615      0.000478046  -0.0031526   0.0   0.122033
 -0.257407     2.41344     -0.000656686  -0.00360535  0.0   0.0439661
  0.00147842  -0.00193047   2.4167        8.59143     0.0  -0.0137771
  2.98994e-5  -0.00260515  -0.257077     -0.500127    0.0   0.00254321
 -0.00937121  -0.0829385   -0.00270552   -0.0152608   1.0  -0.000482959
  0.0          0.0          0.0           0.0         0.0   1.0

In [7]:
# Finite differencing using FiniteDiff.jl:
prep_fdif1 = DI.prepare_jacobian(track_a_particle, AutoFiniteDiff(), x0, Constant(fodo))
J_fdif1 = DI.jacobian(track_a_particle, prep_fdif1, AutoFiniteDiff(), x0, Constant(fodo))

6×6 Matrix{Float64}:
 -0.501416     8.58615      0.000478046  -0.00315261  0.0   0.122033
 -0.257407     2.41344     -0.000656687  -0.00360536  0.0   0.0439661
  0.00147842  -0.00193047   2.4167        8.59143     0.0  -0.0137771
  2.98995e-5  -0.00260515  -0.257077     -0.500127    0.0   0.00254321
 -0.00937118  -0.0829385   -0.00270552   -0.0152609   1.0  -0.000482934
  0.0          0.0          0.0           0.0         0.0   1.0

In [8]:
# Finite differencing using FiniteDifferences.jl:
prep_fdif2 = DI.prepare_jacobian(track_a_particle, AutoFiniteDifferences(;fdm=central_fdm(3,1)), x0, Constant(fodo))
J_fdif2 = DI.jacobian(track_a_particle, prep_fdif2, AutoFiniteDifferences(;fdm=central_fdm(3,1)), x0, Constant(fodo))

6×6 Matrix{Float64}:
 -0.501416     8.58615      0.000478046  -0.0031526   0.0   0.122033
 -0.257407     2.41344     -0.000656686  -0.00360535  0.0   0.0439661
  0.00147842  -0.00193047   2.4167        8.59143     0.0  -0.0137771
  2.98994e-5  -0.00260515  -0.257077     -0.500127    0.0   0.00254321
 -0.00937121  -0.0829385   -0.00270552   -0.0152608   1.0  -0.00048296
  0.0          0.0          0.0           0.0         0.0   1.0

In [9]:
# Show they're all equal:
J_FD ≈ J_GTPSA ≈ J_RD ≈ J_fdif1 ≈ J_fdif2

true